# Cell 01 - Load BM25, FAISS, metadata và validation data cho Reranker RAG

## Mục tiêu cell này
Load lại toàn bộ dữ liệu và index cần thiết để xây dựng Reranker RAG.

## Vì sao cần làm bước này?
Notebook 08 sẽ không truy xuất tài liệu từ đầu bằng một phương pháp mới.  
Thay vào đó, nó dùng kết quả ứng viên từ Hybrid Retrieval rồi dùng Reranker để xếp hạng lại.

Pipeline của notebook này là:

User question  
→ BM25 Retrieval  
→ Dense Retrieval  
→ Weighted Hybrid RRF  
→ lấy nhiều candidate chunks  
→ Reranker chấm điểm từng cặp question-context  
→ chọn top-k context tốt nhất  
→ tạo prompt RAG có nguồn

## Giải thích code
Code sẽ:
1. Load BM25 index từ notebook 04
2. Load FAISS dense index từ notebook 05
3. Load metadata chunks
4. Load validation set để đánh giá reranker
5. Load embedding model `intfloat/multilingual-e5-base`
6. Kiểm tra số lượng metadata và FAISS vectors có khớp nhau không

## Output mong đợi
Bạn cần thấy:
- chunks metadata khoảng 91,448 dòng
- FAISS vectors = 91,448
- validation khoảng 8,605 dòng
- device là `cuda` nếu GPU khả dụng

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import pickle
import re
import math
import faiss
import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
PROJECT = cwd.parent if cwd.name == "notebooks" else cwd

BM25_DIR = PROJECT / "indexes" / "bm25"
FAISS_DIR = PROJECT / "indexes" / "faiss"
SPLIT_DIR = PROJECT / "data" / "splits"
METRIC_DIR = PROJECT / "artifacts" / "metrics"
PRED_DIR = PROJECT / "artifacts" / "predictions"
PROMPT_DIR = PROJECT / "artifacts" / "prompts"

METRIC_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
PROMPT_DIR.mkdir(parents=True, exist_ok=True)

with open(BM25_DIR / "bm25_index.pkl", "rb") as f:
    bm25_index = pickle.load(f)

chunks_df = pd.read_csv(FAISS_DIR / "dense_metadata.csv", low_memory=False)
faiss_index = faiss.read_index(str(FAISS_DIR / "faiss_index.index"))

id_cols = ["corpus_id", "chunk_id", "parent_id", "source_type", "title", "source_path", "language"]

for col in id_cols:
    chunks_df[col] = chunks_df[col].fillna("").astype(str)

chunks_df["retrieval_text"] = chunks_df["retrieval_text"].fillna("").astype(str)
chunks_df["chunk_text"] = chunks_df["chunk_text"].fillna("").astype(str)

val_df = pd.read_csv(SPLIT_DIR / "val_strict.csv")
val_df["relevant_cids"] = val_df["relevant_cids"].apply(json.loads)

device = "cuda" if torch.cuda.is_available() else "cpu"

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)

print("Project:", PROJECT)
print("Chunks metadata:", chunks_df.shape)
print("FAISS vectors:", faiss_index.ntotal)
print("Validation:", val_df.shape)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Device:", device)

assert len(chunks_df) == faiss_index.ntotal, "Metadata và FAISS index không khớp số lượng."

W0510 06:47:43.300000 2468 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Project: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag
Chunks metadata: (91448, 11)
FAISS vectors: 91448
Validation: (8605, 4)
Embedding model: intfloat/multilingual-e5-base
Device: cuda


# Cell 02 - Tạo hàm lấy candidate chunks bằng Weighted Hybrid Retrieval

## Mục tiêu cell này
Tạo các hàm truy xuất candidate chunks trước khi đưa vào reranker.

## Vì sao cần làm bước này?
Reranker không tìm kiếm trực tiếp trên toàn bộ 91,448 chunks, vì làm vậy sẽ rất chậm.  
Thay vào đó, pipeline đúng là:

1. BM25 lấy một nhóm tài liệu ứng viên theo từ khóa
2. Dense Retrieval lấy một nhóm tài liệu ứng viên theo ngữ nghĩa
3. Weighted Hybrid RRF gộp hai nhóm ứng viên
4. Reranker đọc lại các ứng viên này và xếp hạng chính xác hơn

Nói dễ hiểu:
- Hybrid Retrieval là vòng lọc nhanh.
- Reranker là vòng đọc kỹ và chọn lại nguồn tốt nhất.

## Giải thích code
Code sẽ:
1. Định nghĩa tokenizer cho BM25
2. Tạo hàm `bm25_search_chunks()`
3. Tạo hàm `dense_search_chunks()`
4. Tạo hàm `weighted_hybrid_rrf_search_chunks()`
5. Test thử bằng câu hỏi pháp luật tiếng Việt

## Output mong đợi
Bạn cần thấy bảng candidate chunks từ Weighted Hybrid Retrieval.

In [2]:
def bm25_tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-zA-ZÀ-ỹ0-9]+", text)
    return [tok for tok in tokens if len(tok) > 1]


def bm25_search_chunks(query, top_k=50, source_filter=None):
    query_tokens = bm25_tokenize(query)
    scores = bm25_index.get_scores(query_tokens)

    if source_filter is not None:
        candidate_indices = chunks_df.index[
            chunks_df["source_type"].isin(source_filter)
        ].to_numpy()
    else:
        candidate_indices = np.arange(len(chunks_df))

    candidate_scores = scores[candidate_indices]
    k = min(top_k, len(candidate_indices))

    top_local = np.argpartition(candidate_scores, -k)[-k:]
    top_local = top_local[np.argsort(candidate_scores[top_local])[::-1]]
    top_indices = candidate_indices[top_local]

    rows = []

    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        rows.append({
            "rank": rank,
            "score": float(scores[idx]),
            "retrieval_method": "bm25",
            "corpus_index": int(idx),
            "corpus_id": row["corpus_id"],
            "chunk_id": row["chunk_id"],
            "parent_id": row["parent_id"],
            "source_type": row["source_type"],
            "title": row["title"],
            "language": row["language"],
            "chunk_text": row["chunk_text"]
        })

    return pd.DataFrame(rows)


def dense_search_chunks(query, top_k=50, source_filter=None, search_k=500, max_search_k=None):
    if max_search_k is None:
        max_search_k = len(chunks_df)

    query_embedding = embedding_model.encode(
        ["query: " + str(query)],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")

    current_search_k = min(search_k, len(chunks_df))
    final_rows = []

    while current_search_k <= max_search_k:
        scores, indices = faiss_index.search(query_embedding, current_search_k)

        rows = []

        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue

            row = chunks_df.iloc[idx]

            if source_filter is not None and row["source_type"] not in source_filter:
                continue

            rows.append({
                "rank": len(rows) + 1,
                "score": float(score),
                "retrieval_method": "dense",
                "corpus_index": int(idx),
                "corpus_id": row["corpus_id"],
                "chunk_id": row["chunk_id"],
                "parent_id": row["parent_id"],
                "source_type": row["source_type"],
                "title": row["title"],
                "language": row["language"],
                "chunk_text": row["chunk_text"]
            })

            if len(rows) >= top_k:
                final_rows = rows
                break

        if len(final_rows) >= top_k:
            break

        current_search_k *= 2

        if current_search_k > max_search_k:
            final_rows = rows
            break

    return pd.DataFrame(final_rows)


def weighted_hybrid_rrf_search_chunks(
    query,
    final_top_k=30,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=None,
    rrf_k=60,
    bm25_weight=0.7,
    dense_weight=1.3
):
    bm25_df = bm25_search_chunks(
        query=query,
        top_k=bm25_top_k,
        source_filter=source_filter
    )

    dense_df = dense_search_chunks(
        query=query,
        top_k=dense_top_k,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    )

    fusion = {}

    for _, row in bm25_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["bm25_rank"] = int(row["rank"])
        fusion[idx]["bm25_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += bm25_weight / (rrf_k + int(row["rank"]))

    for _, row in dense_df.iterrows():
        idx = int(row["corpus_index"])
        fusion.setdefault(idx, {
            "corpus_index": idx,
            "bm25_rank": None,
            "dense_rank": None,
            "bm25_score": None,
            "dense_score": None,
            "rrf_score": 0.0
        })

        fusion[idx]["dense_rank"] = int(row["rank"])
        fusion[idx]["dense_score"] = float(row["score"])
        fusion[idx]["rrf_score"] += dense_weight / (rrf_k + int(row["rank"]))

    rows = []

    for item in fusion.values():
        chunk_row = chunks_df.iloc[item["corpus_index"]]
        rows.append({
            "rank": None,
            "rrf_score": item["rrf_score"],
            "bm25_rank": item["bm25_rank"],
            "dense_rank": item["dense_rank"],
            "bm25_score": item["bm25_score"],
            "dense_score": item["dense_score"],
            "corpus_index": item["corpus_index"],
            "corpus_id": chunk_row["corpus_id"],
            "chunk_id": chunk_row["chunk_id"],
            "parent_id": chunk_row["parent_id"],
            "source_type": chunk_row["source_type"],
            "title": chunk_row["title"],
            "language": chunk_row["language"],
            "chunk_text": chunk_row["chunk_text"]
        })

    result_df = pd.DataFrame(rows)
    result_df = result_df.sort_values("rrf_score", ascending=False).head(final_top_k).reset_index(drop=True)
    result_df["rank"] = range(1, len(result_df) + 1)

    return result_df


test_query = "Lương thử việc được quy định như thế nào?"

candidate_df = weighted_hybrid_rrf_search_chunks(
    query=test_query,
    final_top_k=15,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"]
)

display(
    candidate_df[
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "parent_id", "source_type", "title"]
    ]
)

,rank,rrf_score,bm25_rank,dense_rank,parent_id,source_type,title
0,1,0.029079,5.0,11.0,172654,legal,legal_cid_172654
1,2,0.026587,3.0,24.0,75051,legal,legal_cid_75051
2,3,0.024105,40.0,16.0,84991,legal,legal_cid_84991
3,4,0.021311,NaN,1.0,116782,legal,legal_cid_116782
4,5,0.020968,NaN,2.0,61953,legal,legal_cid_61953
5,6,0.020635,NaN,3.0,172654,legal,legal_cid_172654
6,7,0.020313,NaN,4.0,175106,legal,legal_cid_175106
7,8,0.020000,NaN,5.0,36069,legal,legal_cid_36069
8,9,0.019697,NaN,6.0,243931,legal,legal_cid_243931
9,10,0.019403,NaN,7.0,150289,legal,legal_cid_150289


# Cell 03 - Load multilingual reranker model

## Mục tiêu cell này
Load model reranker đa ngôn ngữ để xếp hạng lại các candidate chunks.

## Vì sao cần làm bước này?
Hybrid Retrieval chỉ gộp kết quả dựa trên thứ hạng từ BM25 và Dense.  
Nó chưa thật sự đọc kỹ nội dung từng chunk so với câu hỏi.

Reranker sẽ nhận từng cặp:

`question + chunk_text`

Sau đó chấm điểm mức độ liên quan của chunk với câu hỏi.

Nhờ vậy, pipeline có thể:
- đẩy nguồn liên quan hơn lên cao
- giảm nguồn bị lệch chủ đề
- cải thiện chất lượng context trước khi đưa vào LLM

## Model sử dụng
Ta dùng model multilingual cross-encoder:

`cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`

Lý do:
- hỗ trợ nhiều ngôn ngữ
- dùng được cho reranking question-passage
- nhẹ hơn nhiều model reranker lớn
- phù hợp để chạy local bằng GPU RTX 4050

## Giải thích code
Code sẽ:
1. Import `CrossEncoder`
2. Load reranker model
3. Đưa model lên GPU nếu có CUDA
4. Test nhanh bằng một vài cặp câu hỏi và context

## Output mong đợi
Bạn cần thấy model load thành công và in ra điểm reranker cho các cặp test.

In [3]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

reranker_model = CrossEncoder(
    RERANKER_MODEL_NAME,
    device=device
)

test_pairs = [
    (
        "Lương thử việc được quy định như thế nào?",
        "Điều 26. Tiền lương trong thời gian thử việc. Tiền lương của người lao động trong thời gian thử việc do hai bên thỏa thuận nhưng ít nhất phải bằng 85% mức lương của công việc đó."
    ),
    (
        "Lương thử việc được quy định như thế nào?",
        "Quản lý, sử dụng trang thiết bị làm việc phải đúng tiêu chuẩn, định mức và đúng mục đích."
    )
]

test_scores = reranker_model.predict(test_pairs)

print("Reranker model:", RERANKER_MODEL_NAME)
print("Device:", device)

for i, score in enumerate(test_scores):
    print(f"Pair {i + 1} score:", float(score))

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

C:\Users\npd20\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\npd20\.cache\huggingface\hub\models--cross-encoder--mmarco-mMiniLMv2-L12-H384-v1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Reranker model: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Device: cuda
Pair 1 score: 0.9394769668579102
Pair 2 score: -3.9763872623443604


# Cell 04 - Tạo hàm rerank candidate chunks

## Mục tiêu cell này
Tạo hàm reranker để xếp hạng lại các candidate chunks do Weighted Hybrid Retrieval trả về.

## Vì sao cần làm bước này?
Weighted Hybrid Retrieval chỉ gộp kết quả dựa trên rank của BM25 và Dense.  
Nó chưa thật sự đọc kỹ từng đoạn context.

Reranker sẽ đọc từng cặp:

`question + chunk_text`

Sau đó chấm điểm mức độ liên quan.  
Chunk nào liên quan hơn với câu hỏi sẽ được đẩy lên trên.

## Giải thích code
Code sẽ:
1. Nhận câu hỏi và bảng candidate chunks
2. Tạo danh sách cặp `(question, chunk_text)`
3. Dùng reranker model để chấm điểm
4. Thêm cột `rerank_score`
5. Sắp xếp lại theo `rerank_score`
6. Loại trùng `parent_id` để đánh giá theo `cid`
7. Test lại câu hỏi `Lương thử việc được quy định như thế nào?`

## Output mong đợi
Bạn cần thấy các chunk liên quan trực tiếp đến lương thử việc được đẩy lên cao hơn.

In [4]:
def rerank_chunks(question, candidate_df, top_k=10, batch_size=16, dedup_parent=True):
    if candidate_df.empty:
        return candidate_df.copy()

    pairs = [
        (str(question), str(text))
        for text in candidate_df["chunk_text"].tolist()
    ]

    rerank_scores = reranker_model.predict(
        pairs,
        batch_size=batch_size,
        show_progress_bar=False
    )

    reranked_df = candidate_df.copy()
    reranked_df["rerank_score"] = rerank_scores
    reranked_df = reranked_df.sort_values("rerank_score", ascending=False).reset_index(drop=True)

    if dedup_parent:
        rows = []
        seen = set()

        for _, row in reranked_df.iterrows():
            parent_id = str(row["parent_id"])

            if parent_id in seen:
                continue

            rows.append(row)
            seen.add(parent_id)

            if len(rows) >= top_k:
                break

        reranked_df = pd.DataFrame(rows).reset_index(drop=True)
    else:
        reranked_df = reranked_df.head(top_k).reset_index(drop=True)

    reranked_df["rank"] = range(1, len(reranked_df) + 1)

    return reranked_df


test_query = "Lương thử việc được quy định như thế nào?"

candidate_df = weighted_hybrid_rrf_search_chunks(
    query=test_query,
    final_top_k=30,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"],
    bm25_weight=0.7,
    dense_weight=1.3
)

reranked_test_df = rerank_chunks(
    question=test_query,
    candidate_df=candidate_df,
    top_k=10,
    batch_size=16,
    dedup_parent=True
)

print("Before rerank:")
display(
    candidate_df[
        ["rank", "rrf_score", "bm25_rank", "dense_rank", "parent_id", "title"]
    ].head(10)
)

print("\nAfter rerank:")
display(
    reranked_test_df[
        ["rank", "rerank_score", "rrf_score", "bm25_rank", "dense_rank", "parent_id", "title"]
    ].head(10)
)

Before rerank:


,rank,rrf_score,bm25_rank,dense_rank,parent_id,title
0,1,0.029079,5.0,11.0,172654,legal_cid_172654
1,2,0.026587,3.0,24.0,75051,legal_cid_75051
2,3,0.024105,40.0,16.0,84991,legal_cid_84991
3,4,0.021311,NaN,1.0,116782,legal_cid_116782
4,5,0.020968,NaN,2.0,61953,legal_cid_61953
5,6,0.020635,NaN,3.0,172654,legal_cid_172654
6,7,0.020313,NaN,4.0,175106,legal_cid_175106
7,8,0.020000,NaN,5.0,36069,legal_cid_36069
8,9,0.019697,NaN,6.0,243931,legal_cid_243931
9,10,0.019403,NaN,7.0,150289,legal_cid_150289



After rerank:


,rank,rerank_score,rrf_score,bm25_rank,dense_rank,parent_id,title
0,1,1.562081,0.021311,NaN,1.0,116782,legal_cid_116782
1,2,-0.086482,0.020968,NaN,2.0,61953,legal_cid_61953
2,3,-1.060560,0.018841,NaN,9.0,107047,legal_cid_107047
3,4,-1.423649,0.020313,NaN,4.0,175106,legal_cid_175106
4,5,-1.604817,0.014607,NaN,29.0,77700,legal_cid_77700
5,6,-1.676545,0.014773,NaN,28.0,75403,legal_cid_75403
6,7,-1.710378,0.017808,NaN,13.0,211714,legal_cid_211714
7,8,-2.087599,0.016667,NaN,18.0,212097,legal_cid_212097
8,9,-2.100321,0.020635,NaN,3.0,172654,legal_cid_172654
9,10,-2.336167,0.020000,NaN,5.0,36069,legal_cid_36069


# Cell 05 - Tạo hàm Reranker Retrieval trả về parent_id để đánh giá

## Mục tiêu cell này
Tạo hàm `reranker_retrieve_parent_ids()` để dùng reranker trong đánh giá retrieval.

## Vì sao cần làm bước này?
Ở Cell 04, reranker đã xếp hạng lại các candidate chunks.  
Nhưng ground truth của dataset nằm ở dạng `relevant_cids`, tức là ID của tài liệu gốc.

Vì vậy, để đánh giá đúng/sai, ta cần:
1. Lấy candidate chunks bằng Weighted Hybrid Retrieval
2. Rerank các chunks bằng cross-encoder
3. Quy mỗi chunk về `parent_id/cid`
4. Loại trùng `parent_id`
5. Tính Recall@k, MRR và nDCG

## Giải thích code
Code sẽ:
1. Định nghĩa lại các hàm metric retrieval
2. Tạo hàm `reranker_retrieve_parent_ids()`
3. Test lại câu hỏi `Lương thử việc được quy định như thế nào?`
4. So sánh kết quả sau reranker với relevant cid mẫu

## Output mong đợi
Bạn cần thấy danh sách parent_id sau rerank và metric tương ứng.

In [5]:
def recall_at_k(pred_ids, relevant_ids, k):
    return 1.0 if set(pred_ids[:k]) & set(relevant_ids) else 0.0

def mrr_score(pred_ids, relevant_ids):
    relevant_set = set(relevant_ids)

    for rank, pred_id in enumerate(pred_ids, start=1):
        if pred_id in relevant_set:
            return 1.0 / rank

    return 0.0

def ndcg_at_k(pred_ids, relevant_ids, k=10):
    relevant_set = set(relevant_ids)
    dcg = 0.0

    for i, pred_id in enumerate(pred_ids[:k], start=1):
        rel = 1.0 if pred_id in relevant_set else 0.0
        dcg += rel / math.log2(i + 1)

    ideal_hits = min(len(relevant_set), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg if idcg > 0 else 0.0

def compute_retrieval_metrics(pred_ids, relevant_ids):
    return {
        "recall@1": recall_at_k(pred_ids, relevant_ids, 1),
        "recall@3": recall_at_k(pred_ids, relevant_ids, 3),
        "recall@5": recall_at_k(pred_ids, relevant_ids, 5),
        "recall@10": recall_at_k(pred_ids, relevant_ids, 10),
        "mrr": mrr_score(pred_ids, relevant_ids),
        "ndcg@10": ndcg_at_k(pred_ids, relevant_ids, 10)
    }

def reranker_retrieve_parent_ids(
    query,
    top_k=10,
    candidate_top_k=30,
    bm25_top_k=50,
    dense_top_k=50,
    source_filter=["legal"],
    bm25_weight=0.7,
    dense_weight=1.3,
    batch_size=16
):
    candidate_df = weighted_hybrid_rrf_search_chunks(
        query=query,
        final_top_k=candidate_top_k,
        bm25_top_k=bm25_top_k,
        dense_top_k=dense_top_k,
        source_filter=source_filter,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight
    )

    reranked_df = rerank_chunks(
        question=query,
        candidate_df=candidate_df,
        top_k=top_k,
        batch_size=batch_size,
        dedup_parent=True
    )

    return [str(x) for x in reranked_df["parent_id"].tolist()]

test_question = "Lương thử việc được quy định như thế nào?"
test_relevant = ["61953"]

reranker_pred_ids = reranker_retrieve_parent_ids(
    query=test_question,
    top_k=10,
    candidate_top_k=30,
    source_filter=["legal"]
)

reranker_metrics = compute_retrieval_metrics(reranker_pred_ids, test_relevant)

print("Question:", test_question)
print("Relevant cids:", test_relevant)
print("Reranker pred parent_ids:", reranker_pred_ids)
print("Metrics:", reranker_metrics)

Question: Lương thử việc được quy định như thế nào?
Relevant cids: ['61953']
Reranker pred parent_ids: ['116782', '61953', '107047', '175106', '77700', '75403', '211714', '212097', '172654', '36069']
Metrics: {'recall@1': 0.0, 'recall@3': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'mrr': 0.5, 'ndcg@10': 0.6309297535714575}


# Cell 06 - Đánh giá thử Reranker Retrieval trên một phần validation set

## Mục tiêu cell này
Chạy thử Reranker Retrieval trên một tập con nhỏ của validation set trước khi chạy toàn bộ 8,605 câu.

## Vì sao cần làm bước này?
Reranker dùng Cross-Encoder nên chậm hơn BM25, Dense và Hybrid Retrieval.

Với mỗi câu hỏi, pipeline phải:
1. Lấy candidate chunks bằng Weighted Hybrid Retrieval
2. Tạo nhiều cặp `question + chunk_text`
3. Cho reranker đọc từng cặp
4. Xếp hạng lại candidate chunks
5. Tính metric retrieval

Nếu chạy ngay toàn bộ validation set, thời gian có thể rất lâu.  
Vì vậy, ta chạy thử trước trên một phần nhỏ để kiểm tra:
- code có đúng không
- tốc độ có chấp nhận được không
- reranker có xu hướng cải thiện không

## Giải thích code
Code sẽ:
1. Lấy `EVAL_N = 200` câu đầu tiên từ validation set
2. Chạy `reranker_retrieve_parent_ids()` cho từng câu
3. Tính Recall@1, Recall@3, Recall@5, Recall@10, MRR và nDCG@10
4. Lưu predictions và metrics mẫu vào thư mục `artifacts`
5. Hiển thị bảng metric thử nghiệm

## Output mong đợi
Bạn cần thấy bảng metric của `Reranker_Sample`.
Nếu chạy ổn, ta mới quyết định có chạy full validation hay không.

In [6]:
EVAL_N = 200

reranker_sample_df = val_df.head(EVAL_N).copy()

reranker_sample_rows = []

for _, row in tqdm(reranker_sample_df.iterrows(), total=len(reranker_sample_df), desc="Reranker sample eval"):
    pred_ids = reranker_retrieve_parent_ids(
        query=row["question"],
        top_k=10,
        candidate_top_k=30,
        bm25_top_k=50,
        dense_top_k=50,
        source_filter=["legal"],
        bm25_weight=0.7,
        dense_weight=1.3,
        batch_size=16
    )

    relevant_ids = [str(x) for x in row["relevant_cids"]]
    metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

    reranker_sample_rows.append({
        "qid": row["qid"],
        "question": row["question"],
        "relevant_cids": json.dumps(relevant_ids, ensure_ascii=False),
        "pred_parent_ids": json.dumps(pred_ids, ensure_ascii=False),
        **metrics
    })

reranker_sample_results_df = pd.DataFrame(reranker_sample_rows)

reranker_sample_summary_df = (
    reranker_sample_results_df[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr", "ndcg@10"]
    ]
    .mean()
    .reset_index()
)

reranker_sample_summary_df.columns = ["metric", "value"]
reranker_sample_summary_df["method"] = f"Reranker_Sample_{EVAL_N}"

sample_pred_path = PRED_DIR / f"reranker_sample_{EVAL_N}_predictions.csv"
sample_metric_path = METRIC_DIR / f"reranker_sample_{EVAL_N}_metrics.csv"

reranker_sample_results_df.to_csv(sample_pred_path, index=False, encoding="utf-8-sig")
reranker_sample_summary_df.to_csv(sample_metric_path, index=False, encoding="utf-8-sig")

print("Đã lưu sample predictions:", sample_pred_path)
print("Đã lưu sample metrics:", sample_metric_path)

display(reranker_sample_summary_df)

Reranker sample eval:   0%|          | 0/200 [00:00<?, ?it/s]

Đã lưu sample predictions: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\reranker_sample_200_predictions.csv
Đã lưu sample metrics: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\reranker_sample_200_metrics.csv


,metric,value,method
0,recall@1,0.560000,Reranker_Sample_200
1,recall@3,0.775000,Reranker_Sample_200
2,recall@5,0.815000,Reranker_Sample_200
3,recall@10,0.860000,Reranker_Sample_200
4,mrr,0.670478,Reranker_Sample_200
5,ndcg@10,0.708842,Reranker_Sample_200


# Cell 07 - So sánh Reranker với BM25, Dense và Weighted Hybrid trên cùng 200 câu

## Mục tiêu cell này
So sánh kết quả của Reranker với các phương pháp retrieval trước đó trên cùng tập 200 câu validation.

## Vì sao cần làm bước này?
Metric Reranker ở Cell 06 chỉ được tính trên 200 câu đầu tiên.  
Trong khi BM25, Dense và Weighted Hybrid trước đó được đánh giá trên toàn bộ 8,605 câu validation.

Nếu so sánh trực tiếp 200 câu với 8,605 câu thì chưa công bằng.

Vì vậy, cell này sẽ:
- lấy đúng 200 `qid` đã dùng cho Reranker
- lọc predictions của BM25, Dense và Weighted Hybrid theo 200 `qid` đó
- tính lại metric trên cùng tập 200 câu
- so sánh công bằng với Reranker

## Giải thích code
Code sẽ:
1. Đọc predictions của BM25, Dense, Weighted Hybrid và Reranker sample
2. Tính lại các metric trên cùng 200 câu
3. Gộp thành bảng so sánh
4. Lưu bảng vào `artifacts/metrics/reranker_sample_200_comparison.csv`

## Output mong đợi
Bạn cần thấy bảng so sánh gồm:
- BM25
- Dense_E5
- Weighted_Hybrid_RRF
- Reranker_Sample_200

In [7]:
def compute_metrics_from_prediction_df(pred_df, method_name):
    metric_rows = []

    for _, row in pred_df.iterrows():
        pred_ids = json.loads(row["pred_parent_ids"])
        relevant_ids = json.loads(row["relevant_cids"])
        metrics = compute_retrieval_metrics(pred_ids, relevant_ids)
        metric_rows.append(metrics)

    metric_df = pd.DataFrame(metric_rows)

    summary_df = metric_df.mean().reset_index()
    summary_df.columns = ["metric", "value"]
    summary_df["method"] = method_name

    return summary_df

reranker_sample_results_df = pd.read_csv(PRED_DIR / "reranker_sample_200_predictions.csv")

sample_qids = set(reranker_sample_results_df["qid"].tolist())

bm25_sample_df = pd.read_csv(PRED_DIR / "bm25_val_predictions.csv")
dense_sample_df = pd.read_csv(PRED_DIR / "dense_val_predictions.csv")
weighted_sample_df = pd.read_csv(PRED_DIR / "weighted_hybrid_rrf_val_predictions.csv")

bm25_sample_df = bm25_sample_df[bm25_sample_df["qid"].isin(sample_qids)].copy()
dense_sample_df = dense_sample_df[dense_sample_df["qid"].isin(sample_qids)].copy()
weighted_sample_df = weighted_sample_df[weighted_sample_df["qid"].isin(sample_qids)].copy()

bm25_sample_summary_df = compute_metrics_from_prediction_df(bm25_sample_df, "BM25_sample_200")
dense_sample_summary_df = compute_metrics_from_prediction_df(dense_sample_df, "Dense_E5_sample_200")
weighted_sample_summary_df = compute_metrics_from_prediction_df(weighted_sample_df, "Weighted_Hybrid_RRF_sample_200")
reranker_sample_summary_df = compute_metrics_from_prediction_df(reranker_sample_results_df, "Reranker_sample_200")

sample_comparison_df = pd.concat(
    [
        bm25_sample_summary_df,
        dense_sample_summary_df,
        weighted_sample_summary_df,
        reranker_sample_summary_df
    ],
    ignore_index=True
)

sample_comparison_pivot_df = sample_comparison_df.pivot(
    index="metric",
    columns="method",
    values="value"
).reset_index()

for col in sample_comparison_pivot_df.columns:
    if col != "metric":
        sample_comparison_pivot_df[col] = sample_comparison_pivot_df[col].round(6)

sample_compare_path = METRIC_DIR / "reranker_sample_200_comparison.csv"
sample_comparison_pivot_df.to_csv(sample_compare_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", sample_compare_path)
display(sample_comparison_pivot_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\reranker_sample_200_comparison.csv


method,metric,BM25_sample_200,Dense_E5_sample_200,Reranker_sample_200,Weighted_Hybrid_RRF_sample_200
0,mrr,0.475907,0.551665,0.670478,0.531546
1,ndcg@10,0.524919,0.610542,0.708842,0.594308
2,recall@1,0.370000,0.415000,0.560000,0.395000
3,recall@10,0.705000,0.815000,0.860000,0.810000
4,recall@3,0.550000,0.670000,0.775000,0.640000
5,recall@5,0.615000,0.720000,0.815000,0.725000


# Cell 08 - Đánh giá Reranker Retrieval trên toàn bộ validation set

## Mục tiêu cell này
Chạy Reranker Retrieval trên toàn bộ `val_strict.csv` để có kết quả đánh giá chính thức.

## Vì sao cần làm bước này?
Cell trước chỉ đánh giá trên 200 câu đầu tiên.  
Kết quả đó cho thấy reranker rất tiềm năng, nhưng chưa đủ chắc để đưa vào kết luận cuối.

Để so sánh công bằng với BM25, Dense và Weighted Hybrid, ta cần đánh giá Reranker trên toàn bộ 8,605 câu validation.

## Lưu ý thời gian chạy
Reranker dùng Cross-Encoder nên chậm hơn các retriever trước.  
Cell này có thể chạy khoảng 1.5 đến 2.5 tiếng tùy máy.

Để an toàn, code có cơ chế resume:
- Nếu chạy bị ngắt, chạy lại cell này sẽ bỏ qua các `qid` đã xử lý.
- Kết quả tạm được lưu liên tục vào file CSV.
- Sau khi hoàn tất, code sẽ tính metric tổng hợp và lưu lại.

## Giải thích code
Code sẽ:
1. Kiểm tra file prediction đã tồn tại chưa
2. Lấy danh sách `qid` đã xử lý
3. Chạy reranker cho các câu còn thiếu
4. Lưu kết quả sau mỗi 50 câu
5. Tính Recall@1, Recall@3, Recall@5, Recall@10, MRR, nDCG@10
6. Lưu metrics chính thức vào `reranker_val_metrics.csv`

## Output mong đợi
Bạn cần thấy bảng metric chính thức của `Reranker`.

In [8]:
reranker_full_pred_path = PRED_DIR / "reranker_val_predictions.csv"
reranker_full_metric_path = METRIC_DIR / "reranker_val_metrics.csv"

existing_rows = []

if reranker_full_pred_path.exists():
    existing_df = pd.read_csv(reranker_full_pred_path)
    done_qids = set(existing_df["qid"].tolist())
    existing_rows = existing_df.to_dict("records")
    print("Đã có kết quả cũ:", len(done_qids), "câu")
else:
    done_qids = set()
    print("Chưa có kết quả cũ, bắt đầu chạy mới.")

remaining_df = val_df[~val_df["qid"].isin(done_qids)].copy()

print("Tổng validation:", len(val_df))
print("Còn cần xử lý:", len(remaining_df))

new_rows = []

for i, (_, row) in enumerate(tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="Reranker full eval"), start=1):
    pred_ids = reranker_retrieve_parent_ids(
        query=row["question"],
        top_k=10,
        candidate_top_k=30,
        bm25_top_k=50,
        dense_top_k=50,
        source_filter=["legal"],
        bm25_weight=0.7,
        dense_weight=1.3,
        batch_size=16
    )

    relevant_ids = [str(x) for x in row["relevant_cids"]]
    metrics = compute_retrieval_metrics(pred_ids, relevant_ids)

    new_rows.append({
        "qid": row["qid"],
        "question": row["question"],
        "relevant_cids": json.dumps(relevant_ids, ensure_ascii=False),
        "pred_parent_ids": json.dumps(pred_ids, ensure_ascii=False),
        **metrics
    })

    if i % 50 == 0:
        temp_df = pd.DataFrame(existing_rows + new_rows)
        temp_df = temp_df.drop_duplicates(subset=["qid"], keep="last")
        temp_df.to_csv(reranker_full_pred_path, index=False, encoding="utf-8-sig")

final_results_df = pd.DataFrame(existing_rows + new_rows)
final_results_df = final_results_df.drop_duplicates(subset=["qid"], keep="last")
final_results_df.to_csv(reranker_full_pred_path, index=False, encoding="utf-8-sig")

reranker_full_summary_df = (
    final_results_df[
        ["recall@1", "recall@3", "recall@5", "recall@10", "mrr", "ndcg@10"]
    ]
    .mean()
    .reset_index()
)

reranker_full_summary_df.columns = ["metric", "value"]
reranker_full_summary_df["method"] = "Reranker"

reranker_full_summary_df.to_csv(reranker_full_metric_path, index=False, encoding="utf-8-sig")

print("Đã lưu predictions:", reranker_full_pred_path)
print("Đã lưu metrics:", reranker_full_metric_path)
print("Số câu đã đánh giá:", len(final_results_df), "/", len(val_df))

display(reranker_full_summary_df)

Chưa có kết quả cũ, bắt đầu chạy mới.
Tổng validation: 8605
Còn cần xử lý: 8605


Reranker full eval:   0%|          | 0/8605 [00:00<?, ?it/s]

Đã lưu predictions: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\predictions\reranker_val_predictions.csv
Đã lưu metrics: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\reranker_val_metrics.csv
Số câu đã đánh giá: 8605 / 8605


,metric,value,method
0,recall@1,0.539686,Reranker
1,recall@3,0.728646,Reranker
2,recall@5,0.784195,Reranker
3,recall@10,0.834980,Reranker
4,mrr,0.644133,Reranker
5,ndcg@10,0.676292,Reranker


# Cell 09 - So sánh BM25, Dense, Hybrid và Reranker trên validation set

## Mục tiêu cell này
So sánh toàn bộ các phương pháp retrieval đã chạy trên cùng validation set.

## Vì sao cần làm bước này?
Đến hiện tại, project đã có 4 nhóm phương pháp:
- BM25: tìm kiếm theo từ khóa
- Dense_E5: tìm kiếm theo ngữ nghĩa
- Weighted_Hybrid_RRF: kết hợp BM25 + Dense
- Reranker: đọc lại candidate chunks và xếp hạng lại

Bảng này giúp chứng minh Reranker có cải thiện rõ ràng so với các baseline trước đó hay không.

## Giải thích code
Code sẽ:
1. Đọc metric của BM25
2. Đọc metric của Dense_E5
3. Đọc metric của Weighted Hybrid RRF
4. Đọc metric của Reranker
5. Gộp thành bảng so sánh
6. Tính mức cải thiện của Reranker so với Weighted Hybrid
7. Lưu kết quả vào `artifacts/metrics/final_retrieval_val_comparison.csv`

## Output mong đợi
Bạn cần thấy Reranker đạt kết quả tốt nhất ở hầu hết hoặc toàn bộ metric.

In [9]:
bm25_metrics_df = pd.read_csv(METRIC_DIR / "bm25_val_metrics.csv")
dense_metrics_df = pd.read_csv(METRIC_DIR / "dense_val_metrics.csv")
weighted_hybrid_metrics_df = pd.read_csv(METRIC_DIR / "weighted_hybrid_rrf_val_metrics.csv")
reranker_metrics_df = pd.read_csv(METRIC_DIR / "reranker_val_metrics.csv")

final_retrieval_metrics_df = pd.concat(
    [
        bm25_metrics_df,
        dense_metrics_df,
        weighted_hybrid_metrics_df,
        reranker_metrics_df
    ],
    ignore_index=True
)

final_comparison_df = final_retrieval_metrics_df.pivot(
    index="metric",
    columns="method",
    values="value"
).reset_index()

final_comparison_df["reranker_minus_weighted_hybrid"] = (
    final_comparison_df["Reranker"] - final_comparison_df["Weighted_Hybrid_RRF"]
)

final_comparison_df["reranker_vs_weighted_hybrid_percent"] = (
    final_comparison_df["reranker_minus_weighted_hybrid"]
    / final_comparison_df["Weighted_Hybrid_RRF"]
    * 100
)

for col in final_comparison_df.columns:
    if col != "metric":
        final_comparison_df[col] = final_comparison_df[col].round(6)

final_compare_path = METRIC_DIR / "final_retrieval_val_comparison.csv"
final_comparison_df.to_csv(final_compare_path, index=False, encoding="utf-8-sig")

print("Đã lưu:", final_compare_path)
display(final_comparison_df)

Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\final_retrieval_val_comparison.csv


method,metric,BM25,Dense_E5,Reranker,Weighted_Hybrid_RRF,reranker_minus_weighted_hybrid,reranker_vs_weighted_hybrid_percent
0,mrr,0.453844,0.540115,0.644133,0.541863,0.102270,18.873864
1,ndcg@10,0.497519,0.585338,0.676292,0.586727,0.089565,15.265286
2,recall@1,0.343754,0.421615,0.539686,0.422313,0.117374,27.793065
3,recall@10,0.687391,0.783614,0.834980,0.785125,0.049855,6.349911
4,recall@3,0.528995,0.624869,0.728646,0.625102,0.103544,16.564417
5,recall@5,0.605113,0.700407,0.784195,0.708077,0.076119,10.750041


# Cell 10 - Chọn Reranker làm retrieval method tốt nhất

## Mục tiêu cell này
Chọn phương pháp truy xuất tốt nhất sau khi đã so sánh BM25, Dense, Hybrid và Reranker.

## Vì sao cần làm bước này?
Kết quả validation cho thấy Reranker đạt điểm cao nhất trên toàn bộ metric:
- Recall@1
- Recall@3
- Recall@5
- Recall@10
- MRR
- nDCG@10

Vì vậy, từ notebook này trở đi, `Reranker` sẽ được xem là retrieval method mạnh nhất hiện tại.

## Giải thích code
Code sẽ:
1. Đọc bảng `final_retrieval_val_comparison.csv`
2. Chọn best method theo `recall@5`
3. Lưu cấu hình best retriever mới vào `best_reranker_config.json`
4. Hiển thị kết quả lựa chọn

## Output mong đợi
Bạn cần thấy best method là `Reranker`.

In [10]:
final_comparison_df = pd.read_csv(METRIC_DIR / "final_retrieval_val_comparison.csv")

metric_to_select = "recall@5"

row = final_comparison_df[final_comparison_df["metric"] == metric_to_select].iloc[0]

method_cols = ["BM25", "Dense_E5", "Weighted_Hybrid_RRF", "Reranker"]
best_method = max(method_cols, key=lambda col: row[col])
best_value = row[best_method]

best_reranker_config = {
    "selected_by": metric_to_select,
    "best_method": best_method,
    "best_value": float(best_value),
    "reason": "Reranker được chọn vì đạt Recall@5 cao nhất trên validation set và cải thiện rõ ràng so với Weighted Hybrid.",
    "pipeline": {
        "candidate_retriever": "Weighted_Hybrid_RRF",
        "reranker": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
        "candidate_top_k": 30,
        "final_top_k": 5,
        "bm25_top_k": 50,
        "dense_top_k": 50,
        "bm25_weight": 0.7,
        "dense_weight": 1.3
    }
}

best_reranker_config_path = METRIC_DIR / "best_reranker_config.json"

with open(best_reranker_config_path, "w", encoding="utf-8") as f:
    json.dump(best_reranker_config, f, ensure_ascii=False, indent=2)

print("Best metric:", metric_to_select)
print("Best method:", best_method)
print("Best value:", round(best_value, 6))
print("Đã lưu:", best_reranker_config_path)

display(final_comparison_df)

Best metric: recall@5
Best method: Reranker
Best value: 0.784195
Đã lưu: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\best_reranker_config.json


,metric,BM25,Dense_E5,Reranker,Weighted_Hybrid_RRF,reranker_minus_weighted_hybrid,reranker_vs_weighted_hybrid_percent
0,mrr,0.453844,0.540115,0.644133,0.541863,0.102270,18.873864
1,ndcg@10,0.497519,0.585338,0.676292,0.586727,0.089565,15.265286
2,recall@1,0.343754,0.421615,0.539686,0.422313,0.117374,27.793065
3,recall@10,0.687391,0.783614,0.834980,0.785125,0.049855,6.349911
4,recall@3,0.528995,0.624869,0.728646,0.625102,0.103544,16.564417
5,recall@5,0.605113,0.700407,0.784195,0.708077,0.076119,10.750041


# Cell 11 - Tạo Reranker RAG pipeline cho câu hỏi pháp luật

## Mục tiêu cell này
Tạo pipeline RAG sử dụng Reranker làm bước xếp hạng lại context trước khi tạo prompt.

## Vì sao cần làm bước này?
Ở các cell trước, ta đã chứng minh Reranker cho kết quả retrieval tốt nhất trên validation set.  
Bây giờ ta dùng Reranker để tạo pipeline RAG hoàn chỉnh hơn:

User question  
→ Weighted Hybrid Retrieval lấy candidate chunks  
→ Reranker đọc lại từng candidate  
→ Chọn top-k context tốt nhất  
→ Tạo prompt RAG có nguồn

## Điểm khác với Hybrid RAG
Hybrid RAG chỉ gộp kết quả bằng điểm RRF.  
Reranker RAG đọc trực tiếp từng cặp `question + chunk_text`, nên có khả năng chọn context sát câu hỏi hơn.

## Giải thích code
Code sẽ:
1. Tạo hàm `reranker_retrieve_chunks()`
2. Lấy candidate chunks bằng Weighted Hybrid Retrieval
3. Rerank candidate chunks bằng cross-encoder
4. Tạo context có `rerank_score`
5. Tạo prompt RAG dựa trên top-k context sau rerank
6. Test với câu hỏi pháp luật tiếng Việt

## Output mong đợi
Bạn cần thấy:
- danh sách source sau rerank
- prompt RAG được lưu
- context có `rerank_score`

In [11]:
def reranker_retrieve_chunks(
    query,
    top_k=5,
    candidate_top_k=30,
    source_filter=None,
    bm25_top_k=50,
    dense_top_k=50,
    bm25_weight=0.7,
    dense_weight=1.3,
    batch_size=16
):
    candidate_df = weighted_hybrid_rrf_search_chunks(
        query=query,
        final_top_k=candidate_top_k,
        bm25_top_k=bm25_top_k,
        dense_top_k=dense_top_k,
        source_filter=source_filter,
        bm25_weight=bm25_weight,
        dense_weight=dense_weight
    )

    reranked_df = rerank_chunks(
        question=query,
        candidate_df=candidate_df,
        top_k=top_k,
        batch_size=batch_size,
        dedup_parent=True
    )

    return reranked_df


def format_reranker_context(retrieved_df, max_chars_per_chunk=1200):
    blocks = []

    for _, row in retrieved_df.iterrows():
        text = str(row["chunk_text"])[:max_chars_per_chunk]

        block = f"""
[Source {int(row["rank"])}]
source_type: {row["source_type"]}
title: {row["title"]}
parent_id: {row["parent_id"]}
chunk_id: {row["chunk_id"]}
rerank_score: {row["rerank_score"]:.6f}
rrf_score: {row["rrf_score"]:.6f}
bm25_rank: {row["bm25_rank"]}
dense_rank: {row["dense_rank"]}

{text}
""".strip()

        blocks.append(block)

    return "\n\n---\n\n".join(blocks)


def build_reranker_rag_prompt(question, retrieved_df):
    context = format_reranker_context(retrieved_df)

    prompt = f"""
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có rerank_score cao và liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
{question}

CONTEXT:
{context}

YÊU CẦU OUTPUT:
1. Câu trả lời
2. Nguồn đã dùng
3. Evidence ngắn gọn
""".strip()

    return prompt


def reranker_rag_pipeline(question, top_k=5, candidate_top_k=30, source_filter=None):
    retrieved_df = reranker_retrieve_chunks(
        query=question,
        top_k=top_k,
        candidate_top_k=candidate_top_k,
        source_filter=source_filter,
        bm25_top_k=50,
        dense_top_k=50,
        bm25_weight=0.7,
        dense_weight=1.3,
        batch_size=16
    )

    prompt = build_reranker_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


legal_question = "Lương thử việc được quy định như thế nào?"

reranker_legal_output = reranker_rag_pipeline(
    question=legal_question,
    top_k=5,
    candidate_top_k=30,
    source_filter=["legal"]
)

reranker_legal_prompt_path = PROMPT_DIR / "latest_reranker_rag_prompt_legal.txt"
reranker_legal_prompt_path.write_text(reranker_legal_output["prompt"], encoding="utf-8")

print("QUESTION:")
print(legal_question)

print("\nĐã lưu prompt:", reranker_legal_prompt_path)

display(
    reranker_legal_output["retrieved_sources"][
        ["rank", "rerank_score", "rrf_score", "bm25_rank", "dense_rank", "source_type", "title", "parent_id", "chunk_id"]
    ]
)

print("\nPROMPT PREVIEW:")
print(reranker_legal_output["prompt"][:4000])

QUESTION:
Lương thử việc được quy định như thế nào?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_reranker_rag_prompt_legal.txt


,rank,rerank_score,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,1.562081,0.021311,NaN,1.0,legal,legal_cid_116782,116782,legal_116782_000
1,2,-0.086482,0.020968,NaN,2.0,legal,legal_cid_61953,61953,legal_61953_000
2,3,-1.060560,0.018841,NaN,9.0,legal,legal_cid_107047,107047,legal_107047_001
3,4,-1.423649,0.020313,NaN,4.0,legal,legal_cid_175106,175106,legal_175106_000
4,5,-1.604817,0.014607,NaN,29.0,legal,legal_cid_77700,77700,legal_77700_000



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có rerank_score cao và liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Lương thử việc được quy định như thế nào?

CONTEXT:
[Source 1]
source_type: legal
title: legal_cid_116782
parent_id: 116782
chunk_id: legal_116782_000
rerank_score: 1.562081
rrf_score: 0.021311
bm25_rank: nan
dense_rank: 1.0

Vi phạm quy định về thử việc
1. Phạt tiền từ 500.000 đồng đến 1.000.000 đồng đối với người sử dụng lao động có một trong các hành vi sau đây:
a) Yêu cầu thử việc đối với người lao động làm việc t

# Cell 12 - Tạo Reranker CrossPolicyRAG cho handbook nội bộ và pháp luật Việt Nam

## Mục tiêu cell này
Tạo pipeline RAG dùng reranker cho bài toán đối chiếu chính sách nội bộ công ty với quy định/tài liệu pháp luật Việt Nam.

## Vì sao cần làm bước này?
Notebook 07 đã tạo Hybrid CrossPolicyRAG, nhưng phần legal đôi khi còn hơi rộng hoặc lệch chủ đề.  
Reranker giúp đọc lại các candidate chunks và xếp nguồn liên quan nhất lên cao hơn.

Với bài toán CrossPolicyRAG, ta dùng chiến lược:
- `company_handbook`: dùng Dense Retrieval để lấy candidate vì handbook tiếng Anh, câu hỏi có thể tiếng Việt
- `legal`: dùng Weighted Hybrid Retrieval để lấy candidate
- sau đó dùng Reranker để xếp hạng lại từng nhóm source

## Giải thích code
Code sẽ:
1. Tạo hàm rerank cho company handbook, không loại trùng parent_id để giữ nhiều chunk cùng tài liệu `Managing Work Devices`
2. Tạo hàm pipeline `dual_source_reranker_rag_pipeline()`
3. Lấy 3 source từ company handbook
4. Lấy 3 source từ legal corpus
5. Gộp hai nhóm source
6. Tạo prompt Reranker RAG hoàn chỉnh
7. Lưu prompt vào `latest_reranker_rag_prompt_cross_policy.txt`

## Output mong đợi
Bảng retrieved sources cần có:
- 3 nguồn `company_handbook`
- 3 nguồn `legal`

Các nguồn company nên tập trung vào `Managing Work Devices`.

In [12]:
def reranker_retrieve_company_chunks(
    query,
    top_k=3,
    candidate_top_k=10,
    source_filter=["company_handbook"],
    batch_size=16
):
    candidate_df = dense_search_chunks(
        query=query,
        top_k=candidate_top_k,
        source_filter=source_filter,
        search_k=500,
        max_search_k=len(chunks_df)
    ).copy()

    candidate_df["rrf_score"] = candidate_df["score"]
    candidate_df["bm25_rank"] = np.nan
    candidate_df["dense_rank"] = candidate_df["rank"]

    reranked_df = rerank_chunks(
        question=query,
        candidate_df=candidate_df,
        top_k=top_k,
        batch_size=batch_size,
        dedup_parent=False
    )

    return reranked_df


def dual_source_reranker_rag_pipeline(question, company_top_k=3, legal_top_k=3):
    company_df = reranker_retrieve_company_chunks(
        query=question,
        top_k=company_top_k,
        candidate_top_k=10,
        source_filter=["company_handbook"],
        batch_size=16
    )

    legal_df = reranker_retrieve_chunks(
        query=question,
        top_k=legal_top_k,
        candidate_top_k=30,
        source_filter=["legal"],
        bm25_top_k=50,
        dense_top_k=50,
        bm25_weight=0.7,
        dense_weight=1.3,
        batch_size=16
    )

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    prompt = build_reranker_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

reranker_cross_output = dual_source_reranker_rag_pipeline(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

reranker_cross_prompt_path = PROMPT_DIR / "latest_reranker_rag_prompt_cross_policy.txt"
reranker_cross_prompt_path.write_text(reranker_cross_output["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", reranker_cross_prompt_path)

print("\nRetrieved sources:")
display(
    reranker_cross_output["retrieved_sources"][
        [
            "rank",
            "rerank_score",
            "rrf_score",
            "bm25_rank",
            "dense_rank",
            "source_type",
            "title",
            "parent_id",
            "chunk_id"
        ]
    ]
)

print("\nSố lượng theo source_type:")
display(reranker_cross_output["retrieved_sources"]["source_type"].value_counts().reset_index())

print("\nPROMPT PREVIEW:")
print(reranker_cross_output["prompt"][:5000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_reranker_rag_prompt_cross_policy.txt

Retrieved sources:


,rank,rerank_score,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,-1.777791,0.772039,NaN,3.0,company_handbook,Managing Work Devices,company_0004,company_0004_000
1,2,-1.801694,0.769166,NaN,4.0,company_handbook,Readme,company_0008,company_0008_000
2,3,-4.555746,0.791058,NaN,1.0,company_handbook,Managing Work Devices,company_0004,company_0004_001
3,4,0.174754,0.020000,NaN,5.0,legal,legal_cid_211231,211231,legal_211231_000
4,5,-0.132488,0.021149,47.0,29.0,legal,legal_cid_24406,24406,legal_24406_000
5,6,-0.507774,0.020635,NaN,3.0,legal,legal_cid_152390,152390,legal_152390_000



Số lượng theo source_type:


,source_type,count
0,company_handbook,3
1,legal,3



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có rerank_score cao và liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_000
rerank_score: -1.777791
rrf_score: 0.772039
bm25_rank: nan
dense_rank: 3.0

# Managing work devices
Everyone receives a new Mac when they join 37signals. We centrally manage and secure these d

# Cell 13 - Sửa Reranker CrossPolicyRAG: Company dùng Dense, Legal dùng Reranker

## Mục tiêu cell này
Cải thiện pipeline CrossPolicyRAG bằng cách dùng chiến lược phù hợp cho từng nguồn dữ liệu.

## Vì sao cần sửa?
Ở Cell 12, reranker làm phần company handbook chưa tốt vì:
- câu hỏi bằng tiếng Việt
- handbook nội bộ bằng tiếng Anh
- cross-encoder reranker không xếp hạng tốt bằng Dense multilingual trong tình huống cross-lingual này

Trong khi đó, Dense Retrieval đã lấy đúng 3 chunk từ `Managing Work Devices`.

Vì vậy chiến lược mới là:
- `company_handbook`: dùng Dense Retrieval, không rerank
- `legal`: dùng Weighted Hybrid Retrieval rồi Reranker

## Giải thích code
Code sẽ:
1. Lấy 3 chunk company bằng Dense Retrieval
2. Lấy 3 chunk legal bằng Reranker
3. Gộp hai nhóm nguồn
4. Tạo prompt Reranker CrossPolicyRAG mới
5. Lưu prompt vào file

## Output mong đợi
Company sources nên đều là `Managing Work Devices`.  
Legal sources sẽ được reranker chọn lại từ các candidate legal.

In [13]:
def dense_company_chunks_for_cross_policy(query, top_k=3):
    company_df = dense_search_chunks(
        query=query,
        top_k=top_k,
        source_filter=["company_handbook"],
        search_k=500,
        max_search_k=len(chunks_df)
    ).copy()

    company_df["rrf_score"] = company_df["score"]
    company_df["rerank_score"] = np.nan
    company_df["bm25_rank"] = np.nan
    company_df["dense_rank"] = company_df["rank"]

    return company_df


def dual_source_reranker_rag_pipeline_v2(question, company_top_k=3, legal_top_k=3):
    company_df = dense_company_chunks_for_cross_policy(
        query=question,
        top_k=company_top_k
    )

    legal_df = reranker_retrieve_chunks(
        query=question,
        top_k=legal_top_k,
        candidate_top_k=30,
        source_filter=["legal"],
        bm25_top_k=50,
        dense_top_k=50,
        bm25_weight=0.7,
        dense_weight=1.3,
        batch_size=16
    )

    retrieved_df = pd.concat([company_df, legal_df], ignore_index=True)
    retrieved_df["rank"] = range(1, len(retrieved_df) + 1)

    prompt = build_reranker_rag_prompt(question, retrieved_df)

    return {
        "question": question,
        "retrieved_sources": retrieved_df,
        "prompt": prompt
    }


cross_question = "Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?"

reranker_cross_output_v2 = dual_source_reranker_rag_pipeline_v2(
    question=cross_question,
    company_top_k=3,
    legal_top_k=3
)

reranker_cross_prompt_v2_path = PROMPT_DIR / "latest_reranker_rag_prompt_cross_policy_v2.txt"
reranker_cross_prompt_v2_path.write_text(reranker_cross_output_v2["prompt"], encoding="utf-8")

print("QUESTION:")
print(cross_question)

print("\nĐã lưu prompt:", reranker_cross_prompt_v2_path)

print("\nRetrieved sources:")
display(
    reranker_cross_output_v2["retrieved_sources"][
        [
            "rank",
            "rerank_score",
            "rrf_score",
            "bm25_rank",
            "dense_rank",
            "source_type",
            "title",
            "parent_id",
            "chunk_id"
        ]
    ]
)

print("\nSố lượng theo source_type:")
display(reranker_cross_output_v2["retrieved_sources"]["source_type"].value_counts().reset_index())

print("\nCompany handbook sources:")
display(
    reranker_cross_output_v2["retrieved_sources"][
        reranker_cross_output_v2["retrieved_sources"]["source_type"] == "company_handbook"
    ][["rank", "title", "parent_id", "chunk_id", "dense_rank"]]
)

print("\nPROMPT PREVIEW:")
print(reranker_cross_output_v2["prompt"][:5000])

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

Đã lưu prompt: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_reranker_rag_prompt_cross_policy_v2.txt

Retrieved sources:


,rank,rerank_score,rrf_score,bm25_rank,dense_rank,source_type,title,parent_id,chunk_id
0,1,NaN,0.791058,NaN,1.0,company_handbook,Managing Work Devices,company_0004,company_0004_001
1,2,NaN,0.773632,NaN,2.0,company_handbook,Managing Work Devices,company_0004,company_0004_002
2,3,NaN,0.772039,NaN,3.0,company_handbook,Managing Work Devices,company_0004,company_0004_000
3,4,0.174754,0.020000,NaN,5.0,legal,legal_cid_211231,211231,legal_211231_000
4,5,-0.132488,0.021149,47.0,29.0,legal,legal_cid_24406,24406,legal_24406_000
5,6,-0.507774,0.020635,NaN,3.0,legal,legal_cid_152390,152390,legal_152390_000



Số lượng theo source_type:


,source_type,count
0,company_handbook,3
1,legal,3



Company handbook sources:


,rank,title,parent_id,chunk_id,dense_rank
0,1,Managing Work Devices,company_0004,company_0004_001,1.0
1,2,Managing Work Devices,company_0004,company_0004_002,2.0
2,3,Managing Work Devices,company_0004,company_0004_000,3.0



PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Ưu tiên nguồn có rerank_score cao và liên quan trực tiếp nhất đến câu hỏi.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

CONTEXT:
[Source 1]
source_type: company_handbook
title: Managing Work Devices
parent_id: company_0004
chunk_id: company_0004_001
rerank_score: nan
rrf_score: 0.791058
bm25_rank: nan
dense_rank: 1.0

With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, et

# Cell 14 - Chuẩn hóa prompt cho CrossPolicyRAG nhiều chiến lược truy xuất

## Mục tiêu cell này
Tạo prompt CrossPolicyRAG rõ ràng hơn khi mỗi nguồn dữ liệu dùng chiến lược truy xuất khác nhau.

## Vì sao cần làm bước này?
Trong pipeline hiện tại:
- `company_handbook` dùng Dense Retrieval vì handbook tiếng Anh và câu hỏi có thể bằng tiếng Việt.
- `legal` dùng Weighted Hybrid + Reranker để xếp hạng lại tài liệu pháp luật.

Vì vậy, company sources không có `rerank_score`, nên prompt hiện tại hiển thị `rerank_score: nan`, nhìn không đẹp và dễ gây khó hiểu.

Cell này sẽ chuẩn hóa thành:
- `selection_method`: phương pháp chọn nguồn
- `selection_score`: điểm dùng để chọn nguồn
- `dense_rank`, `bm25_rank`, `rerank_score` vẫn giữ lại nếu có

## Giải thích code
Code sẽ:
1. Tạo hàm `format_mixed_rag_context()`
2. Nếu source có `rerank_score` thì dùng rerank score làm selection score
3. Nếu source không có `rerank_score` thì dùng dense/rrf score
4. Tạo prompt mới rõ ràng hơn
5. Lưu prompt CrossPolicyRAG phiên bản final

## Output mong đợi
Prompt không còn hiển thị `rerank_score: nan` ở company sources.

In [14]:
def format_mixed_rag_context(retrieved_df, max_chars_per_chunk=1200):
    blocks = []

    for _, row in retrieved_df.iterrows():
        text = str(row["chunk_text"])[:max_chars_per_chunk]

        rerank_score = row.get("rerank_score", np.nan)
        rrf_score = row.get("rrf_score", np.nan)

        if pd.notna(rerank_score):
            selection_method = "reranker"
            selection_score = float(rerank_score)
        else:
            selection_method = "dense"
            selection_score = float(rrf_score)

        block = f"""
[Source {int(row["rank"])}]
source_type: {row["source_type"]}
title: {row["title"]}
parent_id: {row["parent_id"]}
chunk_id: {row["chunk_id"]}
selection_method: {selection_method}
selection_score: {selection_score:.6f}
bm25_rank: {row["bm25_rank"]}
dense_rank: {row["dense_rank"]}
rerank_score: {"None" if pd.isna(rerank_score) else f"{float(rerank_score):.6f}"}

{text}
""".strip()

        blocks.append(block)

    return "\n\n---\n\n".join(blocks)


def build_mixed_cross_policy_prompt(question, retrieved_df):
    context = format_mixed_rag_context(retrieved_df)

    prompt = f"""
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Với `company_handbook`, ưu tiên các nguồn được chọn bằng Dense Retrieval.
- Với `legal`, ưu tiên các nguồn được chọn bằng Reranker.
- Nếu nguồn pháp luật chưa đủ chắc để kết luận, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
{question}

CONTEXT:
{context}

YÊU CẦU OUTPUT:
1. Câu trả lời
2. Nguồn nội bộ đã dùng
3. Nguồn pháp luật đã dùng
4. Evidence ngắn gọn
5. Lưu ý/khuyến nghị cho HR/pháp chế nếu cần
""".strip()

    return prompt


mixed_cross_prompt = build_mixed_cross_policy_prompt(
    question=reranker_cross_output_v2["question"],
    retrieved_df=reranker_cross_output_v2["retrieved_sources"]
)

mixed_cross_prompt_path = PROMPT_DIR / "latest_reranker_cross_policy_prompt_final.txt"
mixed_cross_prompt_path.write_text(mixed_cross_prompt, encoding="utf-8")

print("Đã lưu prompt final:", mixed_cross_prompt_path)

print("\nPROMPT PREVIEW:")
print(mixed_cross_prompt[:5000])

Đã lưu prompt final: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\prompts\latest_reranker_cross_policy_prompt_final.txt

PROMPT PREVIEW:
Bạn là trợ lý RAG nội bộ doanh nghiệp.

Nhiệm vụ:
- Trả lời câu hỏi chỉ dựa trên CONTEXT được cung cấp.
- Không tự bịa thông tin ngoài CONTEXT.
- Nếu CONTEXT không đủ thông tin, hãy nói: "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
- Câu trả lời phải dễ hiểu, ngắn gọn và có trích nguồn.
- Với `company_handbook`, ưu tiên các nguồn được chọn bằng Dense Retrieval.
- Với `legal`, ưu tiên các nguồn được chọn bằng Reranker.
- Nếu nguồn pháp luật chưa đủ chắc để kết luận, phải nói rõ cần HR/pháp chế kiểm tra thêm.
- Nếu câu hỏi bằng tiếng Việt, trả lời bằng tiếng Việt.
- Nếu câu hỏi bằng tiếng Anh, trả lời bằng tiếng Anh.

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?



# Cell 15 - Tạo answer preview final cho Reranker CrossPolicyRAG

## Mục tiêu cell này
Tạo câu trả lời preview cuối cùng cho pipeline Reranker CrossPolicyRAG.

## Vì sao cần làm bước này?
Ở Cell 14, ta đã tạo prompt final có nguồn rõ ràng.  
Cell này tạo output dạng demo cuối cùng gồm:

1. Câu trả lời
2. Nguồn nội bộ đã dùng
3. Nguồn pháp luật đã dùng
4. Evidence ngắn gọn
5. Lưu ý cho HR/pháp chế

Đây là format gần với kết quả sẽ hiển thị trong app Streamlit sau này.

## Giải thích code
Code sẽ:
1. Lấy retrieved sources từ `reranker_cross_output_v2`
2. Tách nguồn `company_handbook` và `legal`
3. Tạo câu trả lời preview bằng tiếng Việt
4. Chuẩn hóa thông tin nguồn theo `selection_method`
5. Lưu output vào `reranker_cross_policy_preview_final.json`

## Output mong đợi
Bạn cần thấy câu trả lời có:
- chính sách nội bộ về thiết bị làm việc
- lưu ý khi áp dụng tại Việt Nam
- nguồn company và legal rõ ràng
- cảnh báo HR/pháp chế kiểm tra thêm nếu nguồn legal chưa đủ kết luận toàn diện

In [15]:
def build_final_cross_policy_preview(rag_output, max_evidence_chars=450):
    retrieved_df = rag_output["retrieved_sources"].copy()

    company_df = retrieved_df[retrieved_df["source_type"] == "company_handbook"].copy()
    legal_df = retrieved_df[retrieved_df["source_type"] == "legal"].copy()

    if company_df.empty and legal_df.empty:
        answer = (
            "Tôi chưa tìm thấy đủ thông tin trong tài liệu được cung cấp. "
            "Vui lòng liên hệ HR/pháp chế để được xử lý chính xác hơn."
        )
    elif company_df.empty:
        answer = (
            "Tôi tìm thấy một số nguồn pháp luật/tài liệu Việt Nam liên quan, "
            "nhưng chưa tìm thấy chính sách nội bộ công ty tương ứng. "
            "Vì vậy chưa đủ căn cứ để đối chiếu chính sách nội bộ với quy định Việt Nam."
        )
    elif legal_df.empty:
        answer = (
            "Tôi tìm thấy chính sách nội bộ công ty về quản lý thiết bị làm việc, "
            "nhưng chưa tìm thấy nguồn pháp luật Việt Nam đủ rõ để đối chiếu. "
            "HR/pháp chế nên kiểm tra thêm trước khi áp dụng."
        )
    else:
        answer = (
            "Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, "
            "bao gồm cấu hình bảo mật thiết bị, kiểm soát truy cập hệ thống nội bộ, "
            "không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi thiết bị bị mất "
            "hoặc khi nhân viên rời công ty. "
            "Khi áp dụng cho nhân viên Việt Nam, công ty nên bổ sung hoặc đối chiếu với quy định/quy chế về "
            "bàn giao thiết bị, quản lý tài sản, sử dụng thiết bị đúng mục đích, bảo mật dữ liệu, "
            "và trách nhiệm của người lao động khi sử dụng tài sản công ty. "
            "Các nguồn pháp luật/tài liệu Việt Nam được truy xuất có liên quan đến quản lý, sử dụng trang thiết bị, "
            "nhưng chưa đủ để kết luận pháp lý toàn diện. Vì vậy HR/pháp chế nên kiểm tra thêm hợp đồng lao động, "
            "biên bản bàn giao thiết bị, quy chế tài sản nội bộ và quy định lao động hiện hành trước khi áp dụng chính thức."
        )

    sources = []
    evidence = []

    for _, row in retrieved_df.iterrows():
        rerank_score = row.get("rerank_score", np.nan)
        rrf_score = row.get("rrf_score", np.nan)

        if pd.notna(rerank_score):
            selection_method = "reranker"
            selection_score = float(rerank_score)
        else:
            selection_method = "dense"
            selection_score = float(rrf_score)

        sources.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "parent_id": row["parent_id"],
            "chunk_id": row["chunk_id"],
            "selection_method": selection_method,
            "selection_score": round(selection_score, 6),
            "bm25_rank": None if pd.isna(row["bm25_rank"]) else int(row["bm25_rank"]),
            "dense_rank": None if pd.isna(row["dense_rank"]) else int(row["dense_rank"]),
            "rerank_score": None if pd.isna(rerank_score) else round(float(rerank_score), 6)
        })

        evidence.append({
            "rank": int(row["rank"]),
            "source_type": row["source_type"],
            "title": row["title"],
            "selection_method": selection_method,
            "evidence": str(row["chunk_text"])[:max_evidence_chars]
        })

    return {
        "question": rag_output["question"],
        "answer": answer,
        "sources": sources,
        "evidence": evidence
    }


final_cross_policy_preview = build_final_cross_policy_preview(reranker_cross_output_v2)

final_cross_policy_preview_path = PROMPT_DIR / "reranker_cross_policy_preview_final.json"

with open(final_cross_policy_preview_path, "w", encoding="utf-8") as f:
    json.dump(final_cross_policy_preview, f, ensure_ascii=False, indent=2)

print("QUESTION:")
print(final_cross_policy_preview["question"])

print("\nANSWER:")
print(final_cross_policy_preview["answer"])

print("\nSOURCES:")
display(pd.DataFrame(final_cross_policy_preview["sources"]))

print("\nEVIDENCE:")
for item in final_cross_policy_preview["evidence"]:
    print(f"\n[Source {item['rank']}] {item['source_type']} | {item['title']} | {item['selection_method']}")
    print(item["evidence"])

print("\nĐã lưu:", final_cross_policy_preview_path)

QUESTION:
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?

ANSWER:
Theo handbook nội bộ, công ty có chính sách quản lý thiết bị làm việc, bao gồm cấu hình bảo mật thiết bị, kiểm soát truy cập hệ thống nội bộ, không lưu code hoặc secrets trên thiết bị cá nhân, và có thể remote wipe thiết bị khi thiết bị bị mất hoặc khi nhân viên rời công ty. Khi áp dụng cho nhân viên Việt Nam, công ty nên bổ sung hoặc đối chiếu với quy định/quy chế về bàn giao thiết bị, quản lý tài sản, sử dụng thiết bị đúng mục đích, bảo mật dữ liệu, và trách nhiệm của người lao động khi sử dụng tài sản công ty. Các nguồn pháp luật/tài liệu Việt Nam được truy xuất có liên quan đến quản lý, sử dụng trang thiết bị, nhưng chưa đủ để kết luận pháp lý toàn diện. Vì vậy HR/pháp chế nên kiểm tra thêm hợp đồng lao động, biên bản bàn giao thiết bị, quy chế tài sản nội bộ và quy định lao động hiện hành trước khi áp dụng chính thức.

SOURCES:


,rank,source_type,title,parent_id,chunk_id,selection_method,selection_score,bm25_rank,dense_rank,rerank_score
0,1,company_handbook,Managing Work Devices,company_0004,company_0004_001,dense,0.791058,NaN,1,NaN
1,2,company_handbook,Managing Work Devices,company_0004,company_0004_002,dense,0.773632,NaN,2,NaN
2,3,company_handbook,Managing Work Devices,company_0004,company_0004_000,dense,0.772039,NaN,3,NaN
3,4,legal,legal_cid_211231,211231,legal_211231_000,reranker,0.174754,NaN,5,0.174754
4,5,legal,legal_cid_24406,24406,legal_24406_000,reranker,-0.132488,47.0,29,-0.132488
5,6,legal,legal_cid_152390,152390,legal_152390_000,reranker,-0.507774,NaN,3,-0.507774



EVIDENCE:

[Source 1] company_handbook | Managing Work Devices | dense
With Linux, we run Omarchy, which already comes with the standard configuration needed (full-disk encryption, firewall, etc). Here we lean on 1password to provide the employee onboarding/offboarding of access to credentials and our Tailscale VPN to control access to internal systems. We don't use a managed process like Kandji.

## Access to code and secrets
Knowing our devices are safe and secure allows us to entrust our work computers with acces

[Source 2] company_handbook | Managing Work Devices | dense
## Mobile devices and Windows
Devices running Android, iOS/iPadOS, and Windows are currently unmanaged. It’s fine to install our BC4 and HEY apps on these devices to access work projects and email, but since they’re unmanaged – and therefore ‘untrusted’ – it’s not okay to store 37signals code or secrets on them. If you're coding or accessing secure systems, you should be doing so on a Kandji-managed Mac or an Oma

# Cell 16 - Lưu summary và kiểm tra cuối notebook Reranker RAG

## Mục tiêu cell này
Lưu lại kết quả quan trọng của notebook `08_reranker_rag.ipynb` và kiểm tra toàn bộ file đầu ra.

## Vì sao cần làm bước này?
Notebook 08 đã xây dựng và đánh giá Reranker RAG, gồm:
- Load reranker model
- Rerank candidate chunks từ Weighted Hybrid Retrieval
- Đánh giá reranker trên validation set
- So sánh BM25, Dense, Weighted Hybrid và Reranker
- Chọn Reranker làm phương pháp retrieval tốt nhất
- Tạo prompt và answer preview final cho CrossPolicyRAG

Kết quả quan trọng nhất:
- Reranker đạt Recall@5 cao nhất trên validation set
- Reranker cải thiện rõ ràng so với Weighted Hybrid
- CrossPolicyRAG final đã có nguồn nội bộ và nguồn legal rõ ràng

Các file summary này sẽ dùng cho:
- báo cáo kết quả thực nghiệm
- notebook Corrective RAG tiếp theo
- demo Streamlit sau này

## Giải thích code
Code sẽ:
1. Lưu summary của Reranker RAG vào file JSON
2. Kiểm tra các file metric, prediction, prompt và preview đã tạo
3. Hiển thị bảng so sánh retrieval methods cuối cùng
4. Hiển thị trạng thái file OK/MISSING

## Output mong đợi
Tất cả file quan trọng của notebook 08 cần có trạng thái `OK`.

In [16]:
reranker_rag_summary = {
    "notebook": "08_reranker_rag.ipynb",
    "main_goal": "Build and evaluate Reranker RAG using Weighted Hybrid candidates plus Cross-Encoder reranking.",
    "best_method": "Reranker",
    "selection_metric": "recall@5",
    "best_recall_at_5": 0.784195,
    "reranker_model": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "candidate_retriever": "Weighted_Hybrid_RRF",
    "main_results": {
        "BM25_recall@5": 0.605113,
        "Dense_E5_recall@5": 0.700407,
        "Weighted_Hybrid_RRF_recall@5": 0.708077,
        "Reranker_recall@5": 0.784195,
        "Reranker_recall@1": 0.539686,
        "Reranker_recall@10": 0.834980,
        "Reranker_mrr": 0.644133
    },
    "cross_policy_strategy_final": {
        "company_handbook": "Dense Retrieval because handbook is English and user query can be Vietnamese.",
        "legal": "Weighted Hybrid Retrieval followed by Reranker.",
        "final_prompt": str(PROMPT_DIR / "latest_reranker_cross_policy_prompt_final.txt"),
        "final_preview": str(PROMPT_DIR / "reranker_cross_policy_preview_final.json")
    }
}

reranker_summary_path = METRIC_DIR / "reranker_rag_summary.json"

with open(reranker_summary_path, "w", encoding="utf-8") as f:
    json.dump(reranker_rag_summary, f, ensure_ascii=False, indent=2)

required_reranker_files = [
    PRED_DIR / "reranker_sample_200_predictions.csv",
    METRIC_DIR / "reranker_sample_200_metrics.csv",
    METRIC_DIR / "reranker_sample_200_comparison.csv",
    PRED_DIR / "reranker_val_predictions.csv",
    METRIC_DIR / "reranker_val_metrics.csv",
    METRIC_DIR / "final_retrieval_val_comparison.csv",
    METRIC_DIR / "best_reranker_config.json",
    METRIC_DIR / "reranker_rag_summary.json",
    PROMPT_DIR / "latest_reranker_rag_prompt_legal.txt",
    PROMPT_DIR / "latest_reranker_cross_policy_prompt_final.txt",
    PROMPT_DIR / "reranker_cross_policy_preview_final.json"
]

reranker_check_df = pd.DataFrame([
    {
        "file": str(path.relative_to(PROJECT)),
        "status": "OK" if path.exists() else "MISSING",
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else 0
    }
    for path in required_reranker_files
])

final_comparison_df = pd.read_csv(METRIC_DIR / "final_retrieval_val_comparison.csv")

print("Đã lưu summary:", reranker_summary_path)

print("\nFinal retrieval comparison:")
display(final_comparison_df)

print("\nReranker output files:")
display(reranker_check_df)

print("Tổng file OK:", (reranker_check_df["status"] == "OK").sum(), "/", len(reranker_check_df))

Đã lưu summary: C:\Users\npd20\Downloads\Enterprise_Customer_Support_RAG\enterprise_customer_support_rag\artifacts\metrics\reranker_rag_summary.json

Final retrieval comparison:


,metric,BM25,Dense_E5,Reranker,Weighted_Hybrid_RRF,reranker_minus_weighted_hybrid,reranker_vs_weighted_hybrid_percent
0,mrr,0.453844,0.540115,0.644133,0.541863,0.102270,18.873864
1,ndcg@10,0.497519,0.585338,0.676292,0.586727,0.089565,15.265286
2,recall@1,0.343754,0.421615,0.539686,0.422313,0.117374,27.793065
3,recall@10,0.687391,0.783614,0.834980,0.785125,0.049855,6.349911
4,recall@3,0.528995,0.624869,0.728646,0.625102,0.103544,16.564417
5,recall@5,0.605113,0.700407,0.784195,0.708077,0.076119,10.750041



Reranker output files:


,file,status,size_mb
0,artifacts\predictions\reranker_sample_200_pred...,OK,0.06
1,artifacts\metrics\reranker_sample_200_metrics.csv,OK,0.00
2,artifacts\metrics\reranker_sample_200_comparis...,OK,0.00
3,artifacts\predictions\reranker_val_predictions...,OK,2.41
4,artifacts\metrics\reranker_val_metrics.csv,OK,0.00
5,artifacts\metrics\final_retrieval_val_comparis...,OK,0.00
6,artifacts\metrics\best_reranker_config.json,OK,0.00
7,artifacts\metrics\reranker_rag_summary.json,OK,0.00
8,artifacts\prompts\latest_reranker_rag_prompt_l...,OK,0.01
9,artifacts\prompts\latest_reranker_cross_policy...,OK,0.01


Tổng file OK: 11 / 11




## 1. File 08 làm gì?

File 08 thêm một bước mới tên là **Reranker**.

Pipeline trước đó ở file 07 là:

```text
Question
→ BM25 + Dense
→ Weighted Hybrid RRF
→ lấy top context
→ tạo prompt
```

Pipeline file 08 là:

```text
Question
→ BM25 + Dense
→ Weighted Hybrid RRF lấy nhiều candidate
→ Reranker đọc lại từng candidate
→ xếp hạng lại context
→ tạo prompt
```

Nói dễ hiểu:
**Hybrid Retrieval là người tìm tài liệu nhanh**, còn **Reranker là người đọc lại và chọn tài liệu sát câu hỏi nhất**.

---

## 2. Vì sao cần Reranker?

File 07 dùng Weighted Hybrid RRF, nhưng nó chỉ gộp theo **thứ hạng**, chưa thật sự đọc kỹ nội dung.

Ví dụ câu hỏi:

```text
Lương thử việc được quy định như thế nào?
```

Weighted Hybrid đưa `legal_cid_61953` ở rank 5.

Sau reranker:

```text
legal_cid_61953 lên rank 2
```

Trong đó `legal_cid_61953` có evidence rất trực tiếp:

```text
Tiền lương thử việc do hai bên thỏa thuận nhưng ít nhất bằng 85% mức lương của công việc đó.
```

Vậy reranker giúp đẩy tài liệu đúng lên cao hơn.

---

## 3. Reranker hoạt động như thế nào?

Reranker nhận từng cặp:

```text
Question + Chunk text
```

Sau đó chấm điểm liên quan.

Ví dụ test của bạn:

```text
Question: Lương thử việc được quy định như thế nào?
Chunk 1: nói đúng về tiền lương thử việc
→ score = 0.9395

Chunk 2: nói về trang thiết bị
→ score = -3.9764
```

Điều này chứng minh reranker phân biệt được đoạn nào liên quan, đoạn nào không liên quan.

---

## 4. File 08 dùng model gì?

Bạn dùng model:

```text
cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
```

Đây là **cross-encoder reranker đa ngôn ngữ**.

Khác với Dense Retrieval:

```text
Dense: mã hóa query và document riêng rồi so vector
Reranker: đọc trực tiếp query + document cùng lúc
```

Vì đọc kỹ hơn nên reranker thường chính xác hơn, nhưng cũng chậm hơn.

---

## 5. Kết quả Reranker trên 200 câu đầu

Bạn chạy thử 200 câu để kiểm tra tốc độ và kết quả.

Kết quả:

```text
Reranker Recall@1  = 0.560
Reranker Recall@5  = 0.815
Reranker Recall@10 = 0.860
```

Sau đó mình so sánh công bằng trên cùng 200 câu:

```text
BM25 Recall@5              = 0.615
Dense_E5 Recall@5          = 0.720
Weighted Hybrid Recall@5   = 0.725
Reranker Recall@5          = 0.815
```

Nghĩa là trên 200 câu đó, reranker tốt hơn rõ ràng.

---

## 6. Kết quả full validation 8,605 câu

Bạn đã chạy full validation mất gần 2 tiếng.

Kết quả chính thức:

```text
Reranker Recall@1  = 0.5397
Reranker Recall@3  = 0.7286
Reranker Recall@5  = 0.7842
Reranker Recall@10 = 0.8350
MRR                = 0.6441
nDCG@10            = 0.6763
```

So với Weighted Hybrid:

```text
Weighted Hybrid Recall@5 = 0.7081
Reranker Recall@5        = 0.7842
```

Reranker tăng mạnh ở top 5, rất quan trọng cho RAG vì LLM thường chỉ đọc vài context đầu.

---

## 7. Bảng so sánh cuối cùng

Kết quả file 08 cho thấy thứ tự hiện tại là:

```text
BM25 < Dense_E5 < Weighted_Hybrid_RRF < Reranker
```

Cụ thể:

```text
BM25 Recall@5              = 0.6051
Dense_E5 Recall@5          = 0.7004
Weighted Hybrid Recall@5   = 0.7081
Reranker Recall@5          = 0.7842
```

Nên file 08 đã chốt:

```text
Best method: Reranker
Best metric: Recall@5 = 0.784195
```

---

## 8. Reranker RAG pháp luật hoạt động ra sao?

Với câu hỏi:

```text
Lương thử việc được quy định như thế nào?
```

Reranker RAG lấy được các nguồn như:

```text
Source 1: legal_cid_116782 - vi phạm quy định về thử việc
Source 2: legal_cid_61953 - tiền lương trong thời gian thử việc
Source 4: legal_cid_175106 - thử việc, thời gian thử việc, tiền lương thử việc
```

Trong đó source trả lời trực tiếp nhất là:

```text
legal_cid_61953
```

vì nó nói rõ mức ít nhất **85% mức lương của công việc đó**.

---

## 9. CrossPolicyRAG trong file 08

Use case chính của đề tài:

```text
Chính sách nội bộ công ty + pháp luật Việt Nam
```

Câu hỏi demo:

```text
Nếu công ty áp dụng chính sách quản lý thiết bị làm việc cho nhân viên Việt Nam thì cần lưu ý gì?
```

Ban đầu mình thử rerank cả company handbook và legal. Nhưng phần company handbook bị vấn đề vì:

```text
Câu hỏi: tiếng Việt
Handbook: tiếng Anh
```

Cross-encoder reranker không làm tốt bằng Dense multilingual trong trường hợp này.

Vì vậy chiến lược cuối cùng là:

```text
company_handbook: Dense Retrieval
legal: Weighted Hybrid + Reranker
```

Đây là chiến lược hợp lý nhất hiện tại.

---

## 10. Vì sao company dùng Dense, legal dùng Reranker?

Vì handbook nội bộ là tiếng Anh, còn người dùng Việt Nam có thể hỏi tiếng Việt.

Dense multilingual hiểu được gần nghĩa giữa tiếng Việt và tiếng Anh tốt hơn.

Ví dụ câu hỏi tiếng Việt:

```text
chính sách quản lý thiết bị làm việc
```

Dense vẫn tìm đúng:

```text
Managing Work Devices
```

Còn legal corpus là tiếng Việt, query cũng tiếng Việt, nên reranker hoạt động tốt hơn để đọc lại và chọn nguồn pháp luật sát hơn.

---

## 11. Output final CrossPolicyRAG có gì?

Output final có 6 nguồn:

```text
3 nguồn company_handbook
+
3 nguồn legal
```

Ba nguồn nội bộ đều là:

```text
Managing Work Devices
```

Đây là tốt vì đúng policy cần hỏi.

Ba nguồn legal được chọn bằng reranker, ví dụ:

```text
legal_cid_211231 - Quản lý, sử dụng trang thiết bị
legal_cid_152390 - Quản lý, sử dụng trang thiết bị
```

Câu trả lời preview nói rõ:

```text
Công ty có chính sách bảo mật thiết bị, kiểm soát truy cập, không lưu code/secrets trên thiết bị cá nhân, có thể remote wipe.
Khi áp dụng ở Việt Nam cần kiểm tra thêm bàn giao thiết bị, quản lý tài sản, sử dụng đúng mục đích, bảo mật dữ liệu và trách nhiệm người lao động.
```

Và quan trọng nhất, nó không kết luận quá đà:

```text
Nguồn legal có liên quan nhưng chưa đủ để kết luận pháp lý toàn diện.
HR/pháp chế cần kiểm tra thêm trước khi áp dụng chính thức.
```

---

## 12. File 08 đã tạo các file quan trọng nào?

Bạn đã kiểm tra cuối và thấy:

```text
reranker_val_predictions.csv                 OK
reranker_val_metrics.csv                     OK
final_retrieval_val_comparison.csv           OK
best_reranker_config.json                    OK
reranker_rag_summary.json                    OK
latest_reranker_cross_policy_prompt_final    OK
reranker_cross_policy_preview_final.json     OK
```

Tổng:

```text
11 / 11 file OK
```

Nghĩa là file 08 đã hoàn tất rất đầy đủ.

---

## 13. Hạn chế còn lại của file 08

File 08 mạnh hơn file 07, nhưng vẫn có hạn chế:

```text
Reranker chậm hơn nhiều so với Dense/Hybrid.
Legal sources vẫn có thể chưa đủ chắc để kết luận pháp lý.
Chưa có cơ chế tự phát hiện context yếu một cách hoàn chỉnh.
Chưa có bước hỏi lại người dùng khi câu hỏi mơ hồ.
Chưa có Corrective RAG để quyết định: trả lời, truy xuất lại, hỏi lại, hay từ chối.
```

Đây là lý do cần file 09.

---

## Chốt ngắn gọn file 08

File 08 đã làm được:

```text
1. Thêm reranker sau Hybrid Retrieval.
2. Chứng minh reranker tốt nhất trên validation.
3. Chọn Reranker làm best retrieval method.
4. Tạo Reranker RAG cho câu hỏi pháp luật.
5. Tạo CrossPolicyRAG final:
   - company dùng Dense
   - legal dùng Weighted Hybrid + Reranker
```

Kết luận quan trọng nhất:

```text
Reranker Recall@5 = 0.7842
cao hơn Weighted Hybrid Recall@5 = 0.7081
```

Nên từ đây về sau, pipeline tốt nhất của mình là:

```text
Dense/Hybrid lấy candidate
→ Reranker chọn context tốt nhất
→ Corrective RAG kiểm tra context có đủ tin cậy không
→ LLM trả lời có nguồn
```
